[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HesusG/dammg-supervisado/blob/main/notebooks/pre2/pre2.ipynb)

# pre2: la regla contra la recta, en julio

Ayer comparamos dos formas de predecir cuántas bicis salen de una estación en una hora:

- **la regla**: "sale lo mismo que hace una semana, a la misma hora";
- **la recta**: una regresión lineal que aprende de los datos.

Hoy lo calculamos con calma, con las 40 estaciones de MiBici, para responder una pregunta:

> **En julio, ¿quién se equivoca menos, la regla o la recta?**

Vamos en orden: cargar los datos, partir en train, val y test, la regla, la recta, las tres métricas (MAE, RMSE y R²), dos gráficas y la respuesta.

## 0. Preparar

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

sns.set_theme()

In [ ]:
URL = "https://hesusg.github.io/dammg-supervisado/datos/mibici_curso.parquet"
df = pd.read_parquet(URL)

print("filas:", len(df))
print("estaciones:", df["station_id"].nunique())
df.head()

Cada fila es **una estación en una hora**. Las columnas que vamos a usar:

| Columna | Qué es |
|---|---|
| `departures` | bicis que salieron en esa hora: lo que queremos predecir ($y$) |
| `departures_lag_168h` | bicis que salieron hace una semana (168 horas), a la misma hora: nuestra $x$ |
| `particion` | a qué parte pertenece la fila: `train`, `val` o `test` |
| `station_id`, `hour`, `hour_of_day` | la estación, la fecha y hora, y la hora del día |

**¿Qué esperamos ver?** 342,680 filas y 40 estaciones.

## 1. Train, val y test

Partimos **por fecha**, no al azar: entrenamos con el pasado y medimos con el futuro, como pasaría en la vida real.

| Parte | Meses | Para qué se usa |
|---|---|---|
| **train** | octubre a junio | para **aprender**: aquí la recta encuentra su $w$ y su $b$ |
| **val** (julio) | julio | para **escoger**: comparar la regla contra la recta, probar columnas, mover un umbral |
| **test** | agosto y septiembre | para **reportar**: se mira **una sola vez**, al final, cuando ya escogimos |

**¿Cuándo usamos val?** Cada vez que tenemos que decidir algo: ¿la regla o la recta?, ¿qué columnas?, ¿qué grado?, ¿qué $\lambda$? Si decidimos mirando test, test deja de ser "datos nuevos" y el número que reportamos sale demasiado optimista.

In [ ]:
train = ...
julio = ...
test = ...

print("train:", len(train), "filas, de", train["hour"].min(), "a", train["hour"].max())
print("julio:", len(julio), "filas, de", julio["hour"].min(), "a", julio["hour"].max())
print("test: ", len(test), "filas, de", test["hour"].min(), "a", test["hour"].max())

**¿Qué esperamos ver?** train con 254,440 filas (oct 2025 a jun 2026), julio con 29,760 y test con 58,480 (agosto y septiembre). Ninguna fecha se repite entre partes.

## 2. La regla: "lo mismo que hace una semana"

¿Por qué la usamos tanto? Porque es la **referencia** (el *baseline*):

- No aprende nada y no cuesta nada: cualquiera la puede aplicar con una hoja de cálculo.
- Las bicis tienen un ciclo semanal muy fuerte: un lunes a las 8 se parece mucho al lunes anterior a las 8.
- Si un modelo **no le gana a la regla**, no vale la pena usarlo. Por eso, cada vez que entrenemos algo, lo primero es compararlo contra ella.

En código, la predicción de la regla es solo copiar una columna:

In [ ]:
prediccion_regla = ...

**Pista: ¿inducir o deducir?** La regla nos la dieron hecha; nadie la aprendió de los datos. Aplicarla a julio es **deducir**: de la regla al caso.

## 3. La recta: $\hat y = w\,x + b$

Usamos la **misma columna** que la regla, para que la comparación sea justa. La diferencia es que la recta **aprende** $w$ y $b$ con train.

La regla es una recta con $w = 1$ y $b = 0$, puesta a mano. La recta busca los valores que hacen más chico el error al cuadrado en train.

In [ ]:
X_train = train[["departures_lag_168h"]]
y_train = train["departures"]

modelo = LinearRegression()
...

w = modelo.coef_[0]
b = modelo.intercept_
print(f"w = {w:.2f}")
print(f"b = {b:.2f}")

**¿Qué esperamos ver?** $w = 0.70$ y $b = 1.18$. Se leen así:

- $w = 0.70$: **por cada bici que salió hace una semana, la predicción sube 0.70 bicis.** No es un porcentaje: ni "sale el 70 % de las bicis" ni "acierta el 70 % de las veces".
- $b = 1.18$: la **base**, lo que predice cuando hace una semana no salió ninguna.

Ejemplo: la estación 51 el lunes 27 de julio a las 8. Hace una semana salieron 32; la regla dice 32 y la recta dice $0.70 \times 32 + 1.18 \approx 23.6$. En realidad salieron 34.

**Pista: ¿inducir o deducir?** Encontrar $w$ y $b$ a partir de los ejemplos de train es **inducir**: de los casos a la regla. Ya con la recta entrenada, predecir julio es **deducir**.

Ahora predecimos julio. Un detalle: una recta **puede predecir números negativos** (si $b$ fuera negativo, o con otras columnas), y no pueden salir −2 bicis. Por eso **recortamos a 0** lo que salga negativo.

Con esta recta en particular no pasa: $x$ nunca es negativa y $b = 1.18$, así que lo más bajo que predice es 1.18. Lo recortamos de todos modos, porque es la costumbre segura.

In [ ]:
X_julio = julio[["departures_lag_168h"]]
y_julio = julio["departures"]

prediccion_recta = ...
prediccion_recta = prediccion_recta.clip(0)     # lo negativo se vuelve 0

print("la predicción más baja:", prediccion_recta.min().round(2))

## 4. Las tres métricas

El **error** de cada fila es $\hat y - y$: lo que predije menos lo que pasó.

| Métrica | Cómo se calcula | Cómo se lee |
|---|---|---|
| **MAE** | el promedio de los errores **sin signo** | "en promedio me equivoco por tantas bicis". Está en bicis |
| **RMSE** | eleva cada error al cuadrado, promedia y saca raíz | también en bicis, pero **castiga más los errores grandes**: fallar por 10 pesa como 100 |
| **R²** | 1 menos (error al cuadrado del modelo ÷ error al cuadrado de decir siempre el promedio de julio) | 0 = igual que decir siempre el promedio; 1 = perfecto. 0.50 = "le quito la mitad del error al cuadrado" |

Primero el MAE y el RMSE de la regla **a mano**, con pandas, para ver que no hay magia:

In [ ]:
errores = prediccion_regla - y_julio

mae_a_mano = ...
rmse_a_mano = ...

print(f"MAE a mano:  {mae_a_mano:.2f}")
print(f"RMSE a mano: {rmse_a_mano:.2f}")

Ahora con la librería. Siempre se pasa **primero lo que pasó** (`y_julio`) y **después lo que predije**.

In [ ]:
mae_regla = ...
rmse_regla = ...
r2_regla = ...

mae_recta = ...
rmse_recta = ...
r2_recta = ...

print(f"REGLA  MAE = {mae_regla:.2f}   RMSE = {rmse_regla:.2f}   R² = {r2_regla:.2f}")
print(f"RECTA  MAE = {mae_recta:.2f}   RMSE = {rmse_recta:.2f}   R² = {r2_recta:.2f}")

**¿Qué esperamos ver?** Lo mismo que a mano para la regla: MAE 2.30 y RMSE 4.16, y R² 0.41. La recta: MAE 2.35, RMSE 3.84 y R² 0.50.

El objeto `modelo` también trae su R², con `.score()`. Da lo mismo que `r2_score`:

In [ ]:
r2_con_score = ...
print(f"R² con modelo.score: {r2_con_score:.2f}")

## 5. Verlo

Una tabla y una gráfica de barras, una por métrica. Ojo: en MAE y RMSE **más bajo es mejor**; en R², **más alto es mejor**.

In [ ]:
resultados = pd.DataFrame({
    "modelo":  ["regla", "regla", "regla", "recta", "recta", "recta"],
    "metrica": ["MAE", "RMSE", "R²", "MAE", "RMSE", "R²"],
    "valor":   [mae_regla, rmse_regla, r2_regla, mae_recta, rmse_recta, r2_recta],
})

resultados.pivot(index="modelo", columns="metrica", values="valor").round(2)

In [ ]:
sns.catplot(data=resultados, x="modelo", y="valor", col="metrica", kind="bar", sharey=False, height=3.5)
plt.show()

Y un día de julio, para ver qué significa equivocarse por 2 o 3 bicis: la estación 51, el lunes 27 de julio.

In [ ]:
dia = julio[(julio["station_id"] == 51) & (julio["hour"].dt.day == 27)].copy()

dia["regla"] = dia["departures_lag_168h"]
dia["recta"] = modelo.predict(dia[["departures_lag_168h"]]).clip(0)

sns.lineplot(x=dia["hour_of_day"], y=dia["departures"], label="lo que pasó", marker="o")
sns.lineplot(x=dia["hour_of_day"], y=dia["regla"], label="regla")
sns.lineplot(x=dia["hour_of_day"], y=dia["recta"], label="recta")
plt.xlabel("hora del día")
plt.ylabel("bicis que salen")
plt.title("Estación 51, lunes 27 de julio")
plt.show()

**Cómo se lee:** la regla copia la forma de la semana pasada, con picos completos, y a veces acierta exacto y a veces falla por mucho (a las 19 h dijo 35 y salieron 20). La recta es más "tímida": se queda más cerca del promedio, nunca pega exacto en un pico, pero tampoco falla tan feo. De madrugada la recta dice 1.18 aunque no salga nadie: es su $b$.

## 6. La respuesta

| Julio (val) | MAE | RMSE | R² |
|---|---|---|---|
| regla | 2.30 | 4.16 | 0.41 |
| recta | 2.35 | 3.84 | 0.50 |

> **En julio, la regla tiene MAE 2.30 y la recta 2.35**: en promedio, la regla se equivoca un poquito menos. **Pero en RMSE gana la recta** (3.84 contra 4.16) y **en R² también** (0.50 contra 0.41).

¿Cómo puede ser? La regla acierta exacto muchas horas, pero cuando falla, falla por mucho. El MAE cuenta cada bici igual; el RMSE y el R² elevan al cuadrado, así que los errores grandes de la regla pesan muchísimo. La recta reparte el error: falla un poco casi siempre, pero casi nunca por mucho.

**¿Quién gana?** Depende de qué error te cuesta más. Si fallar por 10 bicis es mucho peor que fallar diez veces por 1 (por ejemplo, el camión llega a una estación vacía), conviene la recta. Esta decisión la tomamos con **julio (val)**, no con test.

## 7. Test, una sola vez

Ya escogimos con julio. Ahora sí, el reporte final con agosto y septiembre.

In [ ]:
X_test = test[["departures_lag_168h"]]
y_test = test["departures"]

prediccion_regla_test = ...
prediccion_recta_test = ...

print(f"REGLA  MAE = {mean_absolute_error(y_test, prediccion_regla_test):.2f}   "
      f"RMSE = {root_mean_squared_error(y_test, prediccion_regla_test):.2f}   "
      f"R² = {r2_score(y_test, prediccion_regla_test):.2f}")
print(f"RECTA  MAE = {mean_absolute_error(y_test, prediccion_recta_test):.2f}   "
      f"RMSE = {root_mean_squared_error(y_test, prediccion_recta_test):.2f}   "
      f"R² = {r2_score(y_test, prediccion_recta_test):.2f}")

**¿Qué esperamos ver?** Regla: MAE 2.36, RMSE 4.30, R² 0.44. Recta: MAE 2.47, RMSE 4.06, R² 0.50. Lo mismo que en julio: la regla gana en MAE y la recta en RMSE y R². La conclusión se sostiene con datos que nunca vimos.